# 02 - Gold Layer: Heuristic Feature Engineering

In the Lakehouse Medallion architecture, the Gold Layer represents highly refined, business-level aggregated data ready for machine learning consumption. 

While Phase III of our architecture will utilize Deep Sequence Learning (preserving the 86-week temporal dimension), we must establish a strong, interpretable Tabular Baseline. To do this without losing the temporal "story," we will extract advanced heuristics from the time-series.

1. **Volatility & Shape Extraction:** Calculate Variance and Kurtosis to capture behavioral spikes (mega-prescribing events).
2. **Recent Kinematics (EWMA):** Calculate the Exponentially Weighted Moving Average, heavily weighting the final 12 weeks, as recent behavior is the strongest predictor of future intent.
3. **Cyclic Behavior (FFT):** Apply a Fast Fourier Transform to extract the dominant frequency, identifying doctors who prescribe in recurring cycles.
4. **Consolidation:** Merge these temporal heuristics with static demographics to create a single row per HCP.

In [7]:
import pandas as pd
import numpy as np
from scipy.stats import kurtosis
from scipy.fft import fft
import warnings

# Suppress warnings from scipy if it encounters arrays of all zeros
warnings.filterwarnings('ignore')

# ==========================================
# 1. Data Ingestion & Sequence Formatting
# ==========================================
print("Loading Silver Layer Parquet...")

df = pd.read_parquet('data/silver_layer_longitudinal.parquet')

print(f"Total rows loaded: {len(df):,}")

# CRITICAL MLOPS STEP: Temporal Sorting
# Time-series math (like EWMA and FFT) will fail catastrophically if the data is not chronological.
print("Sorting data chronologically to guarantee temporal integrity...")
df = df.sort_values(by=['NUEVO_ID', 'WEEK_ID']).reset_index(drop=True)

# Updated Core Metrics based on Spearman Correlation Matrix
core_metrics = ['UC_TRX', 'ORAL_TRX', 'BRAND1_NBRX', 'DETAILS', 'SAMPLES', 'DIRECTMAIL']

# Filter only metrics that exist in the dataframe
available_core_metrics = [col for col in core_metrics if col in df.columns]
print(f"Metrics selected for heuristic extraction: {available_core_metrics}")

Loading Silver Layer Parquet...
Total rows loaded: 1,800,066
Sorting data chronologically to guarantee temporal integrity...
Metrics selected for heuristic extraction: ['UC_TRX', 'ORAL_TRX', 'BRAND1_NBRX', 'DETAILS', 'SAMPLES', 'DIRECTMAIL']


In [8]:
# ==========================================
# 2. Advanced Feature Extraction Functions
# ==========================================
print("Defining heuristic mathematical functions...")

def calculate_ewma(series, span=12):
    """
    Calculates the final value of an Exponentially Weighted Moving Average.
    A span of 12 heavily weights the last ~3 months of the 86-week sequence.
    This captures the 'momentum' of the HCP right before the prediction point.
    """
    if len(series) == 0 or series.sum() == 0:
        return 0.0
    # Calculate EWMA and return the very last value (index -1)
    return series.ewm(span=span, adjust=False).mean().iloc[-1]

def calculate_dominant_frequency(series):
    """
    Applies Fast Fourier Transform (FFT) to extract cyclic behavior.
    Ignores the DC component (index 0) which is just the sum of the signal.
    Helps identify HCPs with recurring prescription/marketing cycles.
    """
    if len(series) < 2 or series.sum() == 0:
        return 0.0
    
    # Compute the absolute values of the FFT
    y_fft = np.abs(fft(series.values))
    
    # Remove the DC component (frequency 0) to avoid capturing the mean
    y_fft[0] = 0 
    
    # Return the magnitude of the highest frequency peak
    return np.max(y_fft)

def calculate_kurtosis(series):
    """
    Calculates Fisher's Kurtosis. 
    High kurtosis (>3) indicates extreme, infrequent spikes (mega-prescribing events).
    Low kurtosis indicates consistent, flat behavior.
    """
    if len(series) < 4 or series.sum() == 0:
        return 0.0
    # Fisher=True subtracts 3 to make the normal distribution 0.0
    return kurtosis(series, fisher=True, bias=False)

print("Functions successfully loaded into memory.")

Defining heuristic mathematical functions...
Functions successfully loaded into memory.


In [9]:
# ==========================================
# 3. Mass Feature Extraction Engine
# ==========================================
print("Executing Heuristic Extraction Engine across all HCPs...")
print("Note: This step performs heavy mathematics (FFT, Kurtosis) and might take a few minutes.\n")

# List to store processed dataframes for each metric
processed_features = []

# Group by HCP ID
grouped = df.groupby('NUEVO_ID')

for metric in available_core_metrics:
    print(f"Processing metric: {metric}...")
    
    # Apply our heuristic functions
    agg_df = grouped[metric].agg(
        MEAN=np.mean,
        EWMA=calculate_ewma,
        FFT=calculate_dominant_frequency,
        KURTOSIS=calculate_kurtosis
    ).reset_index()
    
    # Rename columns to clearly identify the parent metric (e.g., 'UC_TRX_FFT')
    agg_df = agg_df.rename(columns={
        'MEAN': f'{metric}_MEAN',
        'EWMA': f'{metric}_EWMA',
        'FFT': f'{metric}_FFT',
        'KURTOSIS': f'{metric}_KURTOSIS'
    })
    
    processed_features.append(agg_df)

# Merge all processed metrics back into a single tabular dataframe
print("\nMerging heuristic features into a single tabular view...")
gold_features = processed_features[0]
for i in range(1, len(processed_features)):
    gold_features = pd.merge(gold_features, processed_features[i], on='NUEVO_ID', how='inner')

# ==========================================
# 4. Static Demographics & Label Preservation
# ==========================================
print("Extracting static demographics and ground-truth labels...")

# Define the static columns we want to preserve (add more if necessary, e.g., 'STATE')
static_cols = ['SPECIALTY', 'IS_LABELED', 'ATSEG']
available_static = [col for col in static_cols if col in df.columns]

# Extract the first chronological instance of these static traits per HCP
static_df = df.groupby('NUEVO_ID')[available_static].first().reset_index()

# Final Merge: Combine Demographics with Temporal Heuristics
gold_layer = pd.merge(static_df, gold_features, on='NUEVO_ID', how='inner')

# ==========================================
# 5. Pipeline Export
# ==========================================
output_file = 'data/gold_heuristic_features.parquet'
gold_layer.to_parquet(output_file, index=False)

print("\n==================================================")
print("SUCCESS: Gold Layer Tabular Baseline Generated!")
print("==================================================")
print(f"Final Data Shape: {gold_layer.shape}")
print(f"File saved as: '{output_file}' ready for Phase.")

Executing Heuristic Extraction Engine across all HCPs...
Note: This step performs heavy mathematics (FFT, Kurtosis) and might take a few minutes.

Processing metric: UC_TRX...
Processing metric: ORAL_TRX...
Processing metric: BRAND1_NBRX...
Processing metric: DETAILS...
Processing metric: SAMPLES...
Processing metric: DIRECTMAIL...

Merging heuristic features into a single tabular view...
Extracting static demographics and ground-truth labels...

SUCCESS: Gold Layer Tabular Baseline Generated!
Final Data Shape: (20931, 28)
File saved as: 'data/gold_heuristic_features.parquet' ready for Phase.
